# Step 10 - retrain with matched budgets, score the test split at train time

This notebook replaces `run_experiment` from the main pipeline. It is self-contained: run it on
its own in a **GPU runtime**. What it fixes, and why each fix matters:

* **one epoch ceiling for both methods** (8, with early stopping and patience 2 epochs). The first
  pass gave LoRA 8/4 epochs and full fine-tuning 3/2, so early stopping fired for 4 of 12 LoRA runs
  and 0 of 12 full-FT runs. At exactly the fractions where LoRA "won", LoRA had converged and full
  fine-tuning was still climbing. That comparison cannot be defended.
* **the test split is scored inside the run**, right after the best checkpoint is restored. No
  checkpoint can go missing later, and nothing is ever selected on test.
* **per-example test logits are saved** to `preds/`, which is what makes McNemar and bootstrap
  significance tests possible afterwards without paying for GPU twice.
* **trainable parameters, wall-clock, peak GPU memory and saved-model size are recorded** - the
  efficiency claims LoRA is actually about, none of which the first pass logged.
* **training checkpoints go to local disk**, not Drive; only the best model, curves, predictions
  and a one-file manifest land on Drive.
* **no silent `resume_from_checkpoint`**, so a rerun is a rerun and numbers stop shifting between
  sessions.

Order: cell 1 (Drive + paths) -> cell 2 (installs, then **Restart runtime** and re-run cell 1) ->
cell 3 (data + metrics, ~5 min: the train split is tokenized here) -> cell 4 (the training code) ->
cell 5 (**pilot: one pair, stop and check the gate**) -> cell 6 (the staged grid) -> cell 7 (zip
the artifacts) -> cell 8 (analysis, no GPU needed).

Safe to re-run: every finished run writes `runs/<run_id>.json` to Drive and is skipped on the next
pass, so a Colab disconnect costs you at most the run in flight. Re-run cell 6 to continue.

In [1]:
# ---- cell 1: Drive + paths ----
from google.colab import drive
drive.mount("/content/drive")

import os
PROJECT_DIR = "/content/drive/MyDrive/thesis_bangla_nli"   # same folder as the main notebook
os.makedirs(PROJECT_DIR, exist_ok=True)
print("project:", PROJECT_DIR)
print("existing run manifests:",
      len(os.listdir(f"{PROJECT_DIR}/runs")) if os.path.exists(f"{PROJECT_DIR}/runs") else 0)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
project: /content/drive/MyDrive/thesis_bangla_nli
existing run manifests: 0
Tesla T4, 15360 MiB


In [2]:
# ---- cell 2: deps, then RESTART RUNTIME ----
!pip install -q torchao --upgrade      # peft's version check crashes without this
!pip install -q peft --upgrade
!pip install -q git+https://github.com/csebuetnlp/normalizer
!pip install -q evaluate
# ---- RESTART RUNTIME NOW (Runtime > Restart session), then re-run cell 1 and
#      continue with cell 3. Do not re-run this cell. ----

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 39.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 18.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00


In [2]:
# ---- cell 3: dataset + metrics (train, validation AND test) ----
# Identical preprocessing to the main notebook: same normalizer, same tokenizer,
# max_length=128, padding to max_length. The test split is tokenized here because
# Step 10 scores it inside every run.
from datasets import load_dataset
from transformers import AutoTokenizer
from normalizer import normalize
import numpy as np
import evaluate

dataset = load_dataset("csebuetnlp/xnli_bn", revision="refs/convert/parquet")
tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")

def normalize_and_tokenize(examples):
    s1 = [normalize(t) for t in examples["sentence1"]]
    s2 = [normalize(t) for t in examples["sentence2"]]
    return tokenizer(s1, s2, truncation=True, padding="max_length", max_length=128)

tokenized_ds = {
    split: dataset[split].map(normalize_and_tokenize, batched=True)
    for split in ["train", "validation", "test"]
}

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=labels,
                                      average="macro")["f1"],
    }

print({k: len(v) for k, v in tokenized_ds.items()})
# expected: train 381449, validation 2419, test 4895

xnli_bn/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 74.6MB            

xnli_bn/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/242k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/480k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

... (5 outputs left out)

Map:   0%|          | 0/381449 [00:00<?, ? examples/s]

Map:   0%|          | 0/2419 [00:00<?, ? examples/s]

Map:   0%|          | 0/4895 [00:00<?, ? examples/s]

{'train': 381449, 'validation': 2419, 'test': 4895}


In [3]:
# ---- cell 4: the training code (matched budgets, test scored in-run) ----
# =============================================================================
# STEP 10 - retrain with MATCHED budgets and score the TEST split at train time
# =============================================================================
# Replaces run_experiment(). Every defect this fixes, in order:
#   * both methods get the SAME epoch ceiling and the same early-stopping patience,
#     so neither is cut off while still improving
#   * the held-out test split is scored inside the run, so no checkpoint can ever
#     go missing again and no selection happens on test
#   * per-example test logits are saved, which is what makes McNemar / bootstrap
#     significance tests possible later without paying for GPU twice
#   * trainable parameters, wall-clock, peak GPU memory and saved-model size are
#     recorded - the efficiency claims LoRA is actually about
#   * training checkpoints go to local disk, not Drive; only the best model,
#     curves, predictions and a manifest land on Drive
#   * no silent resume_from_checkpoint, so a rerun is a rerun
#
# Needs from earlier cells: PROJECT_DIR, tokenized_ds (train/validation/test),
#                           compute_metrics
# Writes: runs/<run_id>.json (source of truth), results_v2.csv (derived),
#         curves_v2/<run_id>.json, preds/<run_id>_test.npz, best_models/<run_id>/
# =============================================================================

import glob, json, math, os, shutil, time
import numpy as np
import pandas as pd
import torch
import transformers, peft
from transformers import (AutoModelForSequenceClassification, EarlyStoppingCallback,
                          Trainer, TrainingArguments, default_data_collator, set_seed)
from peft import LoraConfig, TaskType, get_peft_model

# ---- configuration ----------------------------------------------------------
RESULTS_V2   = f"{PROJECT_DIR}/results_v2.csv"
RUNS_DIR     = f"{PROJECT_DIR}/runs"          # one manifest per run; the CSV is
CURVES_DIR   = f"{PROJECT_DIR}/curves_v2"     # rebuilt from these, so a wiped CSV
PREDS_DIR    = f"{PROJECT_DIR}/preds"         # costs nothing
MODELS_DIR   = f"{PROJECT_DIR}/best_models"
SCRATCH_CKPT = "/content/ckpt"                # NOT on Drive

BASE_MODEL      = "csebuetnlp/banglabert"
NUM_LABELS      = 3
BATCH_SIZE      = 16                 # same as your first pass, keep it
EVAL_BATCH      = 64
CEILING         = 8                  # SAME for both methods - the whole point
PATIENCE_EPOCHS = 2                  # in epochs, converted to evals below
PRECISION       = "fp32"             # fp32 like your first pass. fp16 is ~2x faster
                                     # but is a new confound: never change this
                                     # halfway through the grid.
LR       = {"lora": 2e-4, "full_ft": 2e-5}
LORA_CFG = dict(r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["query", "value"])

# PLACEHOLDER_A


def evals_per_epoch(fraction):
    """One selection point per epoch is plenty when an epoch is short; at 25% of
    the data and above an epoch is thousands of steps, so evaluate twice per epoch
    or the best-checkpoint grid is coarser than the differences you are measuring."""
    return 1 if fraction <= 0.10 else 2


KEEP = ["input_ids", "attention_mask", "token_type_ids", "label"]

def _strip(ds):
    return ds.remove_columns([c for c in ds.column_names if c not in KEEP])

def _dir_mb(path):
    total = 0
    for root, _, files in os.walk(path):
        total += sum(os.path.getsize(os.path.join(root, f)) for f in files)
    return round(total / 1e6, 2)

def _mkdirs():
    for d in (RUNS_DIR, CURVES_DIR, PREDS_DIR, MODELS_DIR, SCRATCH_CKPT):
        os.makedirs(d, exist_ok=True)

def build_model(method, seed):
    """Returns (model, trainable, total). set_seed first so the classifier head and
    the LoRA A/B matrices are initialised reproducibly."""
    set_seed(seed)
    base = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=NUM_LABELS)
    if method == "lora":
        model = get_peft_model(base, LoraConfig(task_type=TaskType.SEQ_CLS, **LORA_CFG))
    else:
        model = base
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return model, trainable, total

def rebuild_results_csv():
    """results_v2.csv is DERIVED from runs/*.json - never edit it by hand."""
    rows = [json.load(open(p)) for p in sorted(glob.glob(f"{RUNS_DIR}/*.json"))]
    if not rows:
        print("no run manifests yet")
        return pd.DataFrame()
    df = pd.DataFrame(rows).sort_values(["fraction", "seed", "method"])
    df.to_csv(RESULTS_V2, index=False)
    return df

def done_runs():
    return {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f"{RUNS_DIR}/*.json")}

# PLACEHOLDER_B


def run_one(method, fraction, seed, ceiling=CEILING, force=False):
    """One run, start to finish: train -> select on dev -> score test -> log. ~1 GPU."""
    run_id = f"{method}_frac{fraction}_seed{seed}"
    manifest_path = f"{RUNS_DIR}/{run_id}.json"
    if os.path.exists(manifest_path) and not force:
        print(f"[SKIP] {run_id} already done.")
        return json.load(open(manifest_path))

    _mkdirs()
    set_seed(seed)
    n_train = int(len(tokenized_ds["train"]) * fraction)
    # identical subset for both methods at a given seed, nested as fraction grows
    train_ds = _strip(tokenized_ds["train"].shuffle(seed=seed).select(range(n_train)))
    dev_ds, test_ds = _strip(tokenized_ds["validation"]), _strip(tokenized_ds["test"])

    steps_per_epoch = max(1, math.ceil(n_train / BATCH_SIZE))
    per_epoch = evals_per_epoch(fraction)
    eval_every = max(1, steps_per_epoch // per_epoch)
    patience = PATIENCE_EPOCHS * per_epoch          # patience counted in epochs
    by_steps = per_epoch > 1

    model, trainable, total = build_model(method, seed)
    print(f"[RUN] {run_id}  n_train={n_train}  ceiling={ceiling} epochs  "
          f"lr={LR[method]}  eval every {eval_every} steps  patience={patience} evals  "
          f"trainable={trainable:,}/{total:,} ({100*trainable/total:.2f}%)")

    args = TrainingArguments(
        output_dir=f"{SCRATCH_CKPT}/{run_id}",
        num_train_epochs=ceiling,
        learning_rate=LR[method],
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH,
        eval_strategy="steps" if by_steps else "epoch",
        save_strategy="steps" if by_steps else "epoch",
        **({"eval_steps": eval_every, "save_steps": eval_every} if by_steps else {}),
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        seed=seed,
        data_seed=seed,
        fp16=(PRECISION == "fp16"),
        bf16=(PRECISION == "bf16"),
        logging_steps=max(10, eval_every // 2),
        report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                      eval_dataset=dev_ds, compute_metrics=compute_metrics,
                      data_collator=default_data_collator,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)])

# PLACEHOLDER_C

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_out = trainer.train()                      # no resume, ever
    wall = time.time() - t0
    peak_mb = round(torch.cuda.max_memory_allocated() / 1e6, 1) \
        if torch.cuda.is_available() else None

    # ---- what happened, epoch by epoch (saved so a capped run is diagnosable) ---
    curve = [l for l in trainer.state.log_history if "eval_f1_macro" in l]
    if not curve:
        raise RuntimeError(f"[{run_id}] no eval_f1_macro in log_history - "
                           "compute_metrics key mismatch, fix before logging.")
    best = max(curve, key=lambda x: x["eval_f1_macro"])
    json.dump(curve, open(f"{CURVES_DIR}/{run_id}.json", "w"), indent=2)
    evals_ran = len(curve)
    evals_possible = ceiling * per_epoch
    early_stopped = evals_ran < evals_possible
    ceiling_hit = abs(float(best["epoch"]) - ceiling) < 1e-6

    # ---- dev at the loaded best model: must reproduce the best curve point ------
    dev = trainer.evaluate(dev_ds, metric_key_prefix="dev")
    drift = abs(dev["dev_f1_macro"] - best["eval_f1_macro"])
    if drift > 1e-3:
        print(f"  [!] dev F1 at the restored best model ({dev['dev_f1_macro']:.4f}) "
              f"differs from the best curve point ({best['eval_f1_macro']:.4f}) by "
              f"{drift:.4f} - load_best_model_at_end did not restore what you think.")

    # ---- test: scored ONCE, never used for any choice --------------------------
    pred = trainer.predict(test_ds, metric_key_prefix="test")
    np.savez_compressed(f"{PREDS_DIR}/{run_id}_test.npz",
                        logits=pred.predictions.astype(np.float16),
                        labels=np.asarray(pred.label_ids))

    # ---- the best model itself, on Drive, in a stable place --------------------
    save_dir = f"{MODELS_DIR}/{run_id}"
    shutil.rmtree(save_dir, ignore_errors=True)
    trainer.save_model(save_dir)

# PLACEHOLDER_D

    manifest = {
        "run_id": run_id, "method": method, "fraction": fraction, "seed": seed,
        "n_train": n_train, "epoch_ceiling": ceiling, "best_epoch": float(best["epoch"]),
        "best_step": int(best.get("step", 0)), "evals_ran": evals_ran,
        "evals_possible": evals_possible, "early_stopped": bool(early_stopped),
        "ceiling_hit": bool(ceiling_hit), "patience_evals": patience,
        "patience_epochs": PATIENCE_EPOCHS, "evals_per_epoch": per_epoch,
        "learning_rate": LR[method], "batch_size": BATCH_SIZE, "precision": PRECISION,
        "dev_accuracy": float(dev["dev_accuracy"]), "dev_f1_macro": float(dev["dev_f1_macro"]),
        "dev_f1_from_curve": float(best["eval_f1_macro"]), "dev_reload_drift": float(drift),
        "test_accuracy": float(pred.metrics["test_accuracy"]),
        "test_f1_macro": float(pred.metrics["test_f1_macro"]),
        "trainable_params": int(trainable), "total_params": int(total),
        "pct_trainable": round(100 * trainable / total, 4),
        "train_runtime_s": round(wall, 1),
        "train_samples_per_s": round(train_out.metrics.get("train_samples_per_second", 0), 2),
        "peak_gpu_mem_mb": peak_mb, "saved_model_mb": _dir_mb(save_dir),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "transformers": transformers.__version__, "peft": peft.__version__,
        "torch": torch.__version__,
        "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    json.dump(manifest, open(manifest_path, "w"), indent=2)
    rebuild_results_csv()

    print(f"  dev F1 {manifest['dev_f1_macro']:.4f} @ epoch {manifest['best_epoch']:g}"
          f"{'  (early stopped)' if early_stopped else ''}"
          f"{'  (CEILING HIT - budget may still be binding)' if ceiling_hit else ''}"
          f"  ->  TEST acc {manifest['test_accuracy']:.4f} / F1 "
          f"{manifest['test_f1_macro']:.4f}   [{wall/60:.1f} min, "
          f"{peak_mb} MB peak, model {manifest['saved_model_mb']} MB]")

    del model, trainer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    shutil.rmtree(f"{SCRATCH_CKPT}/{run_id}", ignore_errors=True)   # free local disk
    return manifest


def run_grid(fractions, seeds, methods=("full_ft", "lora"), ceiling=CEILING, force=False):
    """Cheapest fractions first, so a disconnect costs you the least. Every finished
    run is on Drive before the next one starts, so just re-run this cell to continue."""
    todo = [(f, s, m) for f in sorted(fractions) for s in seeds for m in methods]
    print(f"{len(todo)} run(s) queued; {len(done_runs())} already on Drive\n")
    for f, s, m in todo:
        try:
            run_one(m, f, s, ceiling=ceiling, force=force)
        except Exception as e:                        # one bad run must not kill the grid
            print(f"[FAILED] {m}_frac{f}_seed{s}: {type(e).__name__}: {e}")
    return rebuild_results_csv()

### Cell 5 is a gate, not a formality

Run **one pair only** - fraction 0.01, seed 42, both methods - and read the output before you
launch anything else. Each run is a few minutes at this fraction. Pass conditions:

1. dev macro-F1 lands in the neighbourhood of your first pass (full FT ~0.706, LoRA ~0.727 at this
   fraction and seed). A wildly different number means the preprocessing or the label mapping
   changed, not that the method changed.
2. a TEST accuracy and F1 are printed on the same line - that is the number your thesis will report.
3. `runs/`, `curves_v2/`, `preds/` and `best_models/` all gained a file for each run, and
   `results_v2.csv` has two rows.
4. trainable parameters read ~110M for full FT and ~0.44M (0.40%) for LoRA. If LoRA reports 110M,
   `get_peft_model` did not wrap the model and you are timing the wrong thing.
5. wall-clock and peak memory are printed. Multiply the minutes by the number of queued runs before
   committing to the grid.

If any of those fail, stop and fix it. A broken pilot repeated 20 times is 20 wasted GPU hours.

In [4]:
# ---- cell 5: PILOT - one pair, then stop and read the gate above ----
pilot = [run_one("full_ft", 0.01, 42), run_one("lora", 0.01, 42)]

import pandas as pd
cols = ["run_id", "best_epoch", "evals_ran", "early_stopped", "ceiling_hit",
        "dev_f1_macro", "test_accuracy", "test_f1_macro", "trainable_params",
        "pct_trainable", "train_runtime_s", "peak_gpu_mem_mb", "saved_model_mb"]
print("\n" + pd.DataFrame(pilot)[cols].to_string(index=False))
print("\nminutes per run:", [round(p["train_runtime_s"] / 60, 1) for p in pilot])

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[RUN] full_ft_frac0.01_seed42  n_train=3814  ceiling=8 epochs  lr=2e-05  eval every 239 steps  patience=2 evals  trainable=110,619,651/110,619,651 (100.00%)


model.safetensors: reconstructing file:   0%|          |  0.00B /  443MB            

... (17 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.934637,0.849455,0.618437,0.618187
2,0.760514,0.835989,0.644068,0.641450
3,0.707715,0.734570,0.689541,0.692057
4,0.632837,0.742032,0.705663,0.706775
5,0.595260,0.738885,0.714758,0.716536
6,0.561475,0.760345,0.718479,0.720518
7,0.520807,0.763255,0.715172,0.716496
8,0.503028,0.768096,0.716825,0.718343


[transformers] early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.503028,0.760345,8,0.718479,0.720518


  dev F1 0.7205 @ epoch 6  ->  TEST acc 0.7124 / F1 0.7140   [10.3 min, 1543.5 MB peak, model 3.57 MB]

                 run_id  best_epoch  evals_ran  early_stopped  ceiling_hit  dev_f1_macro  test_accuracy  test_f1_macro  trainable_params  pct_trainable  train_runtime_s  peak_gpu_mem_mb  saved_model_mb
full_ft_frac0.01_seed42         5.0          7           True        False      0.706073       0.721553        0.72230         110619651       100.0000            788.7           2674.3          442.51
   lora_frac0.01_seed42         6.0          8          False        False      0.720518       0.712360        0.71402            887811         0.7962            620.4           1543.5            3.57

minutes per run: [13.1, 10.3]


### Cell 6: the grid, cheapest first

Order matters because a disconnect should cost you the least. Cell 6 runs in stages and each stage
is independently reportable, so you can stop after any of them and still have a results chapter:

* **stage 1** - fractions 0.01 and 0.05, both methods, 5 seeds. This is the decisive experiment:
  the low-data regime is where the crossover claim lives, and 5 seeds is what turns a 1-point gap
  into evidence instead of noise. Roughly 20 runs, all cheap.
* **stage 2** - fraction 0.10, both methods, 5 seeds. The fraction where the first pass reported
  LoRA's largest win.
* **stage 3, optional** - fraction 1.00, one or two seeds. Confirms the high-data end at a matched
  ceiling; each run is expensive, so treat it as a bonus and lower the ceiling if you must.

Fractions 0.25 and 0.50 stay dev-only from the first pass and must be labelled as such in the
thesis. Uncomment stages as you go; already-finished runs are skipped instantly, so re-running the
whole cell after a disconnect is the intended way to resume.

In [ ]:
# ---- cell 6: the staged grid - re-run this cell to resume after a disconnect ----
SEEDS = [42, 7, 13, 21, 33]

# stage 1: the decisive low-data experiment (~20 runs, cheap)
df = run_grid([0.01, 0.05], SEEDS)

# stage 2: uncomment once stage 1 is complete
# df = run_grid([0.10], SEEDS)

# stage 3 (optional, expensive): full data, fewer seeds, same ceiling
# df = run_grid([1.00], [42])

import pandas as pd
pd.set_option("display.width", 220)
print(f"\n{len(df)} runs on Drive")
print(df.groupby(["fraction", "method"])[["dev_f1_macro", "test_f1_macro"]]
        .agg(["mean", "std", "count"]).round(4).to_string())

In [ ]:
# ---- cell 7: one zip with everything the analysis needs ----
# Small (manifests, curves, predictions) - the best_models/ dirs are excluded.
import shutil, os
STAGE = "/content/step10_artifacts"
shutil.rmtree(STAGE, ignore_errors=True)
os.makedirs(STAGE, exist_ok=True)
for name in ["runs", "curves_v2", "preds", "figures"]:
    src = f"{PROJECT_DIR}/{name}"
    if os.path.exists(src):
        shutil.copytree(src, f"{STAGE}/{name}")
for f in ["results_v2.csv", "significance.csv", "results_log.csv"]:
    if os.path.exists(f"{PROJECT_DIR}/{f}"):
        shutil.copy(f"{PROJECT_DIR}/{f}", STAGE)
zip_path = shutil.make_archive(f"{PROJECT_DIR}/step10_artifacts", "zip", STAGE)
print(zip_path, round(os.path.getsize(zip_path) / 1e6, 2), "MB")
from google.colab import files
files.download(zip_path)

### Cell 8: analysis - no GPU, no training

Reads `runs/*.json` and `preds/*.npz` and prints, in order: consistency checks (anything that would
make the comparison unfair), test macro-F1 by fraction and method, paired per-seed gaps, McNemar
plus a paired bootstrap CI, the efficiency table, and two figures. Run it on a CPU runtime if you
like - just re-run cell 1 first so `PROJECT_DIR` exists.

Read the consistency block first. If it flags a ceiling still binding, or seeds that differ between
methods, the tables below it are not yet a fair comparison and you should fix that before quoting
any number.

In [ ]:
# ---- cell 8: analysis, tables, significance, figures ----
# =============================================================================
# STEP 11 - analysis: test-set tables, significance tests, efficiency, figures
# =============================================================================
# Reads what Step 10 wrote (runs/*.json + preds/*.npz). No GPU, no training.
# Produces, in order:
#   1. consistency checks - anything that would make the comparison unfair
#   2. TEST macro-F1 by fraction and method, mean +/- sd over seeds
#   3. paired per-seed gaps (the honest way to compare when seeds are shared)
#   4. McNemar exact test on accuracy + paired bootstrap CI on macro-F1
#   5. efficiency table: trainable params, wall-clock, peak memory, model size
#   6. figures into figures/
# Needs from earlier cells: PROJECT_DIR (and Step 10's constants if run in the
# same session; otherwise the paths below are self-contained)
# =============================================================================

import glob, itertools, json, os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from scipy.stats import binomtest

RUNS_DIR   = f"{PROJECT_DIR}/runs"
PREDS_DIR  = f"{PROJECT_DIR}/preds"
FIG_DIR    = f"{PROJECT_DIR}/figures"
OLD_LOG    = f"{PROJECT_DIR}/results_log.csv"      # the first pass, dev-only
BOOT_N     = 2000                                   # bootstrap resamples
os.makedirs(FIG_DIR, exist_ok=True)

runs = [json.load(open(p)) for p in sorted(glob.glob(f"{RUNS_DIR}/*.json"))]
if not runs:
    raise SystemExit("no runs/*.json yet - run Step 10 first")
df = pd.DataFrame(runs).sort_values(["fraction", "method", "seed"]).reset_index(drop=True)
print(f"{len(df)} runs: "
      + ", ".join(f"{f:g}={n}" for f, n in df.groupby('fraction').size().items()))

# PLACEHOLDER_A

# ---- 1. consistency checks --------------------------------------------------
print("\n=== consistency ===")
problems = []
# these must be identical everywhere, or the two methods were not run under the
# same rules. patience_evals is NOT one of them: it is patience_epochs x evals per
# epoch, so it legitimately differs between fractions - check it per fraction.
for col in ["precision", "batch_size", "epoch_ceiling", "patience_epochs"]:
    if col in df.columns and df[col].nunique() > 1:
        problems.append(f"{col} differs across runs: {sorted(df[col].unique())} - "
                        f"runs are not comparable unless this is deliberate")
for frac, g in df.groupby("fraction"):
    for col in ["patience_evals", "evals_per_epoch"]:
        if col in g.columns and g[col].nunique() > 1:
            problems.append(f"fraction {frac:g}: {col} differs within the fraction "
                            f"({sorted(g[col].unique())}) - selection grids differ")
for (frac, method), g in df.groupby(["fraction", "method"]):
    if g["seed"].duplicated().any():
        problems.append(f"duplicate seeds at fraction {frac}, {method}")
for frac, g in df.groupby("fraction"):
    per = g.groupby("method")["seed"].apply(set)
    if len(per) < 2:
        problems.append(f"fraction {frac}: only {list(per.index)} present - no comparison")
    elif len(per) == 2 and per.iloc[0] != per.iloc[1]:
        problems.append(f"fraction {frac}: seeds differ between methods "
                        f"({per.iloc[0]} vs {per.iloc[1]}) - gaps are not paired")
missing_preds = [r for r in df["run_id"] if not os.path.exists(f"{PREDS_DIR}/{r}_test.npz")]
if missing_preds:
    problems.append(f"no saved predictions for {missing_preds} - no significance test")
hit = df[df["ceiling_hit"]]
if len(hit):
    problems.append(f"ceiling still binding for {list(hit['run_id'])} - the budget "
                    f"limited these runs, so say so or raise the ceiling")
drift = df[df["dev_reload_drift"] > 1e-3]
if len(drift):
    problems.append(f"best model did not reload cleanly for {list(drift['run_id'])}")
print("\n".join(f"  [!] {p}" for p in problems) if problems else "  nothing to flag")
print(f"  early stopped: {int(df['early_stopped'].sum())}/{len(df)} "
      f"({dict(df.groupby('method')['early_stopped'].sum())})")

# ---- 2. test scores by fraction and method ----------------------------------
print("\n=== TEST macro-F1 (mean +/- sd over seeds) ===")
agg = (df.groupby(["fraction", "method"])[["test_f1_macro", "test_accuracy"]]
         .agg(["mean", "std", "count"]).round(4))
print(agg.to_string())

# PLACEHOLDER_B

# ---- 3. paired per-seed gaps -------------------------------------------------
# Both methods share the seed's data subset, so the per-seed difference is a
# paired measurement: report those, not just two independent means.
print("\n=== paired gaps, LoRA minus full FT (per seed) ===")
wide = df.pivot_table(index=["fraction", "seed"], columns="method",
                      values="test_f1_macro")
pairs = wide.dropna(subset=[c for c in ("lora", "full_ft") if c in wide.columns])
if {"lora", "full_ft"}.issubset(pairs.columns):
    pairs = pairs.assign(gap=(pairs["lora"] - pairs["full_ft"]).round(4))
    print(pairs.round(4).to_string())
    summary = pairs.groupby("fraction")["gap"].agg(["mean", "std", "count",
                                                    lambda s: int((s > 0).sum())])
    summary.columns = ["mean_gap", "sd", "n_seeds", "seeds_lora_ahead"]
    print("\n" + summary.round(4).to_string())
else:
    print("  need both methods at the same fraction+seed to pair")

# ---- 4. significance --------------------------------------------------------
def load_preds(run_id):
    z = np.load(f"{PREDS_DIR}/{run_id}_test.npz")
    return z["logits"].astype(np.float32).argmax(1), z["labels"]

def mcnemar(a_pred, b_pred, y):
    """exact test on per-example correctness: does either method win more often?"""
    a, b_ = (a_pred == y), (b_pred == y)
    b = int((a & ~b_).sum())          # first method right, second wrong
    c = int((~a & b_).sum())
    p = binomtest(b, b + c, 0.5).pvalue if (b + c) else 1.0
    return b, c, p

def boot_f1_gap(a_pred, b_pred, y, n=BOOT_N, seed=0):
    """paired bootstrap over test examples: 95% CI for the macro-F1 difference.
    McNemar tests accuracy; your headline metric is macro-F1, so test that too."""
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(y), size=(n, len(y)))
    d = np.array([f1_score(y[i], a_pred[i], average="macro")
                  - f1_score(y[i], b_pred[i], average="macro") for i in idx])
    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

# PLACEHOLDER_C

print("\n=== significance, per fraction and seed (LoRA vs full FT) ===")
print("  b = LoRA right / full FT wrong, c = the reverse; p = exact McNemar.")
sig_rows = []
for (frac, seed), g in df.groupby(["fraction", "seed"]):
    have = dict(zip(g["method"], g["run_id"]))
    if not {"lora", "full_ft"}.issubset(have):
        continue
    if any(not os.path.exists(f"{PREDS_DIR}/{have[m]}_test.npz") for m in ("lora", "full_ft")):
        continue
    lp, y = load_preds(have["lora"])
    fp, y2 = load_preds(have["full_ft"])
    assert np.array_equal(y, y2), "test labels differ between runs - wrong preds file"
    b, c, p = mcnemar(lp, fp, y)
    f1_l = f1_score(y, lp, average="macro")
    f1_f = f1_score(y, fp, average="macro")
    lo, hi = boot_f1_gap(lp, fp, y)
    sig_rows.append({"fraction": frac, "seed": seed, "f1_lora": round(f1_l, 4),
                     "f1_full_ft": round(f1_f, 4), "f1_gap": round(f1_l - f1_f, 4),
                     "gap_ci95": f"[{lo:+.4f}, {hi:+.4f}]",
                     "ci_excludes_0": (lo > 0) or (hi < 0),
                     "mcnemar_b": b, "mcnemar_c": c, "mcnemar_p": round(p, 4),
                     "sig_05": p < 0.05})
sig = pd.DataFrame(sig_rows)
if len(sig):
    print(sig.to_string(index=False))
    sig.to_csv(f"{PROJECT_DIR}/significance.csv", index=False)
    n_sig = int(sig["sig_05"].sum())
    print(f"\n  {n_sig} of {len(sig)} seed-level comparisons reach p<0.05 on accuracy; "
          f"{int(sig['ci_excludes_0'].sum())} have a macro-F1 CI that excludes zero.")
    print("  A gap that fails both is noise: say 'no measurable difference', not "
          "'LoRA was slightly better'.")
else:
    print("  no paired prediction files yet")

# ---- 5. efficiency ----------------------------------------------------------
print("\n=== efficiency (what LoRA is actually for) ===")
eff = (df.groupby("method")
         .agg(trainable_params=("trainable_params", "max"),
              pct_trainable=("pct_trainable", "max"),
              saved_model_mb=("saved_model_mb", "mean"),
              peak_gpu_mem_mb=("peak_gpu_mem_mb", "mean"),
              samples_per_s=("train_samples_per_s", "mean")).round(2))
print(eff.to_string())
print("\nwall-clock minutes per run, by fraction:")
print((df.pivot_table(index="fraction", columns="method", values="train_runtime_s",
                      aggfunc="mean") / 60).round(1).to_string())

# PLACEHOLDER_D

# ---- 6. figures -------------------------------------------------------------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for method, style in (("full_ft", "o-"), ("lora", "s--")):
    g = df[df["method"] == method].groupby("fraction")["test_f1_macro"]
    if not len(g):
        continue
    ax.errorbar(g.mean().index, g.mean().values,
                yerr=g.std().fillna(0).values, fmt=style, capsize=3,
                label="full fine-tuning" if method == "full_ft" else "LoRA")
ax.set_xscale("log")
ax.set_xlabel("fraction of training data (log scale)")
ax.set_ylabel("test macro-F1")
ax.set_title("Matched epoch budget, selection on validation, reported on test")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(f"{FIG_DIR}/test_f1_vs_fraction.{ext}", dpi=200)
print(f"\nwrote {FIG_DIR}/test_f1_vs_fraction.png/.pdf")

# dev curves for the matched runs: shows WHY the old comparison was unfair
fig2, ax2 = plt.subplots(figsize=(6, 4))
for _, r in df.iterrows():
    cp = f"{PROJECT_DIR}/curves_v2/{r['run_id']}.json"
    if not os.path.exists(cp):
        continue
    c = json.load(open(cp))
    ax2.plot([e["epoch"] for e in c], [e["eval_f1_macro"] for e in c],
             ("s--" if r["method"] == "lora" else "o-"), alpha=0.7,
             color=("tab:orange" if r["method"] == "lora" else "tab:blue"),
             label=r["method"] if r["run_id"].endswith(str(df["seed"].iloc[0])) else None)
ax2.set_xlabel("epoch")
ax2.set_ylabel("validation macro-F1")
ax2.set_title("Convergence at a matched ceiling")
ax2.grid(alpha=0.3)
handles, labels = ax2.get_legend_handles_labels()
if labels:
    ax2.legend(dict(zip(labels, handles)).values(), dict(zip(labels, handles)).keys())
fig2.tight_layout()
for ext in ("png", "pdf"):
    fig2.savefig(f"{FIG_DIR}/dev_curves_matched.{ext}", dpi=200)
print(f"wrote {FIG_DIR}/dev_curves_matched.png/.pdf")

# ---- the old first pass, for the fractions you are not retraining ------------
if os.path.exists(OLD_LOG):
    old = pd.read_csv(OLD_LOG)
    keep = old[~old["fraction"].isin(df["fraction"].unique())]
    if len(keep):
        print("\n=== first pass, DEV only - label these clearly as dev, unmatched "
              "budgets, 2 seeds ===")
        print(keep.pivot_table(index="fraction", columns="method",
                               values="f1_macro", aggfunc="mean").round(4).to_string())

### How to report this

**The headline is whatever the paired gaps and the significance table say together.** For each
fraction you have a mean per-seed gap, a count of how many seeds LoRA led on, a McNemar p-value on
accuracy and a bootstrap CI on macro-F1. If the CI includes zero and p > 0.05, the honest sentence
is "no measurable difference at this data size", not "LoRA was slightly better". Report the gap and
its interval either way - a well-bounded null result is a real finding, and it is a much stronger
thesis than an unsupported crossover.

**Say what changed and why.** The first pass compared converged LoRA against full fine-tuning that
was cut off while still improving, and reported the maximum of a validation curve as if it were a
test score. Step 10 gives both methods the same ceiling and the same patience, selects on
validation, and reports on the untouched test split. Both are in your appendix; only the second is
in your results chapter.

**Lead the efficiency table.** LoRA trains ~0.4% of the parameters and saves a checkpoint two
orders of magnitude smaller. That claim survives regardless of how the accuracy comparison lands,
and it is the reason anyone uses LoRA. Quote trainable parameters, saved-model MB, peak GPU memory
and wall-clock from the table cell 8 prints.

**State the limits plainly:** fractions 0.25 and 0.50 are first-pass validation numbers under
unmatched budgets, fp32 throughout, one LoRA configuration (r=8, alpha=16, query+value), one
learning rate per method, and no learning-rate search - so any claim is about these settings, not
about the methods in general.

Evidence and reasoning: `THESIS_FIX_LIST.md`. Task order: `THESIS_TODO.pdf`.